# __FUND_CODE__ – __ANALYSE__

*Beskriv hvad analysen skal svare på, og hvem der skal læse den.*

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from pe_analysis import data, metrics
from pe_analysis import style as ps
from pe_analysis.style import (
    PPIM_SAGE_DARK, PPIM_SAGE_MID, PPIM_SAGE_LIGHT, PPIM_OLIVE, PPIM_BROWN, PPIM_TAUPE,
    PPIM_TAUPE_LIGHT, PPIM_TERRACOTTA, PPIM_CREAM, PPIM_BEIGE, PPIM_RED_MUTED, PPIM_RED_LIGHT,
    SLIDE, MX, BOTTOM_FRAC, INK, LINE, GRID, style_ax, add_header, add_footer, source_line,
    ref_pill, ref_pill_v, legend_swatches, save, _dk, money, mult, render_table,
)
from pe_analysis.deck import byg_deck

## Konfiguration

In [ ]:
FUND_CODE = "__FUND_CODE__"
PRAEFIKS  = f"{FUND_CODE.lower()}___ANALYSE__"   # præfiks på alle filer i charts/ og exports/

Path("charts").mkdir(exist_ok=True)
Path("exports").mkdir(exist_ok=True)
C = lambda navn: str(Path("charts") / f"{PRAEFIKS}_{navn}.png")

## Indlæsning

Se `analysis-data-guide.md` for viewene. Investorniveau (`cash_flows`, `nav`) og fondsniveau
(`fund_metrics`, `holdings`, `breakdown`) må ikke lægges sammen.

In [ ]:
fond = data.fond(FUND_CODE)
holdings = data.holdings(FUND_CODE)
REPORT_DATE = holdings["report_date"].max()

# Husstilen læser fond, dato og valuta som modulattributter (sidehoved, kildelinje, money())
ps.FUND = fond["fund_name"]
ps.REPORT_DK = f"{REPORT_DATE:%d.%m.%Y}"
ps.VALUTA = fond["currency"]
ps.KILDE = "PE-databasen, GP'ens fondsrapporter"
print(f"{FUND_CODE}: {len(holdings)} beholdningslinjer, seneste rapport {ps.REPORT_DK}")

## 1. Første graf

Mønsteret for et slide: figur i 16:9, sidehoved, layout inden for sidehovedet, kildelinje, `save`.

In [ ]:
d = (holdings[holdings["report_date"] == REPORT_DATE].groupby("name")["fair_value"].sum()
     .sort_values(ascending=False).head(10)[::-1] / 1e6)

fig, ax = plt.subplots(figsize=SLIDE)
style_ax(ax)
top = add_header(fig, "De ti største beholdninger", f"Dagsværdi i mio. {ps.VALUTA} · hele fonden")
ax.barh(d.index, d.values, color=PPIM_SAGE_DARK)
fig.tight_layout(rect=[MX, BOTTOM_FRAC, 1 - MX, top])
add_footer(fig, source_line(), ax=ax)
save(fig, C("stoerste"))
plt.show()

## Præsentation

Højst syv sektioner (agenda-layoutet har syv rækker). Tallene i tekstslides regnes ud, de skrives
ikke af.

In [ ]:
byg_deck(
    f"exports/{PRAEFIKS}_{REPORT_DATE:%d%m%Y}.pptx",
    titel=FUND_CODE, undertitel=f"__ANALYSE__ pr. {ps.REPORT_DK}",
    sektioner=[{"titel": "Beholdninger", "slides": [C("stoerste")]}],
)